# Missingness Mechanism Analysis

This notebook analyzes the patterns of missing values in the dataset to determine if they are:
- **MCAR** (Missing Completely At Random)
- **MAR** (Missing At Random)
- **MNAR** (Missing Not At Random)

Understanding this helps in choosing the best imputation or handling strategy.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add pipeline directory to path
sys.path.append(os.path.abspath('.'))

from pipeline.feature_engineering import (
    RawFeatureGenerator,
    RollingStatFeatureGenerator,
    GroupedFeatureGenerator,
    MomentumGenerator,
    VolatilityRatioGenerator,
    InteractionGenerator,
    ShortTermInteractionGenerator
)

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load Sample Data
X_train = pd.read_csv('Data/X_train_sample.csv')
y_train = pd.read_csv('Data/y_train_sample.csv')

train_df = X_train.merge(y_train, on='ROW_ID')
y = (train_df['target'] > 0).astype(int)
X = train_df.drop(columns=['target', 'ROW_ID'])

print(f"Data Shape: {X.shape}")

## 1. Generate Features (to introduce NaNs)
We use the existing pipeline which generates NaNs naturally (e.g. rolling windows).

In [ ]:
lag_features = [f'RET_{i}' for i in range(1, 21)]
volume_features = [f'SIGNED_VOLUME_{i}' for i in range(1, 21)]

generators = [
    RawFeatureGenerator(cols=lag_features + volume_features + ['MEDIAN_DAILY_TURNOVER']),
    RollingStatFeatureGenerator(cols=lag_features, windows=[5, 10, 20], operations=['mean', 'std', 'min', 'max']),
    MomentumGenerator(windows=[(5, 20), (1, 5), (1, 20)]),
    VolatilityRatioGenerator(windows=[(5, 20)]),
    GroupedFeatureGenerator(group_col='GROUP', target_cols=['RET_1', 'SIGNED_VOLUME_1'], operations=['mean']),
    InteractionGenerator(),
    ShortTermInteractionGenerator(max_lag=10)
]

X_transformed = pd.DataFrame(index=X.index)
for gen in generators:
    gen.fit(X, y)
    X_part = gen.transform(X)
    X_transformed = pd.concat([X_transformed, X_part], axis=1)

# Remove duplicates
X_transformed = X_transformed.loc[:, ~X_transformed.columns.duplicated()]

print(f"Generated Features: {X_transformed.shape[1]}")

## 2. Visualize Missingness

In [ ]:
# Calculate missing % per column
missing_percent = X_transformed.isnull().mean() * 100
missing_cols = missing_percent[missing_percent > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 6))
missing_cols.plot(kind='bar')
plt.title('Percentage of Missing Values by Feature')
plt.ylabel('Percent Missing (%)')
plt.show()

print("Top missing features:")
print(missing_cols.head(10))

## 3. Test for MNAR (Missing Not At Random)
We check if the *presence* of a missing value is correlated with the **Target**.
If the missingness is strongly correlated with the target, it suggests MNAR (or at least that missingness is informative).

In [ ]:
# Create a binary matrix: 1 if missing, 0 if present
missing_indicator = X_transformed[missing_cols.index].isnull().astype(int)

# Calculate correlation with Target
correlations = {}
for col in missing_indicator.columns:
    corr = np.corrcoef(missing_indicator[col], y)[0, 1]
    correlations[col] = corr

corr_series = pd.Series(correlations).sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 6))
corr_series.head(20).plot(kind='barh')
plt.title('Correlation between Missingness and Target (MNAR Check)')
plt.xlabel('Correlation Coefficient')
plt.show()

print("Top correlations between Missingness and Target:")
print(corr_series.head(10))

### Interpretation
- **High Correlation** -> **Informative Missingness (likely MNAR)**. The fact that data is missing predicts the target. In this case, creating a separate "is_missing" feature or using tree-based models (like LightGBM) that handle NaNs as a specific branch is best.
- **Low Correlation** -> Could be MCAR or MAR. 

Since LightGBM handles missing values by learning the best direction for NaNs, if missingness is informative (correlated with target), LightGBM will exploit it naturally.